In [1]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [2]:
%sql spark

In [3]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio

time="2026-05-20T20:19:16+05:30" level=warning msg="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container minio Creating 
 Container minio Created 
 Container minio Starting 
 Container minio Started 


In [3]:
%%bash

## Delete Existing Delta Table /invoices_cow
aws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake/lc_ex1 --recursive



CalledProcessError: Command 'b'\n## Delete Existing Delta Table /invoices_cow\naws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake/lc_ex1 --recursive\n\n'' returned non-zero exit status 1.

In [4]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/lc_ex1 --recursive

In [ ]:
df1 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df2 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet")
df3 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")

In [6]:
df_union = df1.union(df2).union(df3)
df_union = df_union.select("customer_id", "category", "price", "quantity", "invoice_date")

In [7]:
%%sql 

DROP TABLE IF EXISTS spark_catalog.deltalake.lc_ex1;


Running query in 'SparkSession'

++
||
++
++

In [8]:

#
df_union.write.mode("overwrite").partitionBy("invoice_date").format("delta").saveAsTable("spark_catalog.deltalake.lc_ex1")

In [9]:
%%sql

OPTIMIZE spark_catalog.deltalake.lc_ex1
ZORDER BY customer_id; 

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/lc_ex1,"Row(numFilesAdded=797, numFilesRemoved=985, filesAdded=Row(min=2111, max=2540, avg=2329.7779171894604, totalFiles=797, totalSize=1856833), filesRemoved=Row(min=1215, max=2540, avg=2121.9543147208124, totalFiles=985, totalSize=2090125), partitionsOptimized=797, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=985, size=2090125), inputNumCubes=0, mergedFiles=Row(num=985, size=2090125), numOutputCubes=797, mergedNumCubes=None), clusteringStats=None, numBins=797, numBatches=1, totalConsideredFiles=985, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1779289309474, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [3]:
%%sql

SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex1; 

Running query in 'SparkSession'

1 rows affected.

Field 1
99457


In [ ]:

-- Create an empty table
CREATE TABLE table1(col0 int, col1 string) USING DELTA CLUSTER BY (col0);

-- Using a CTAS statement (Delta 3.3+)
CREATE EXTERNAL TABLE table2 CLUSTER BY (col0)  -- specify clustering after table name, not in subquery
LOCATION 'table_location'
AS SELECT * FROM table1;

In [6]:
%%sql 

CREATE TABLE IF NOT EXISTS spark_catalog.deltalake.lc_ex2 USING DELTA CLUSTER BY (invoice_date, customer_id)
AS SELECT * FROM spark_catalog.deltalake.lc_ex1



Running query in 'SparkSession'

++
||
++
++

In [24]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.deltalake.lc_ex2

Running query in 'SparkSession'

13 rows affected.

Field 1,Field 2,Field 3
customer_id,int,None
category,string,None
price,double,None
quantity,int,None
invoice_date,date,None
,,
# Detailed Table Information,,
Name,spark_catalog.deltalake.lc_ex2,
Type,MANAGED,
Location,s3a://warehouse/default/deltalake/lc_ex2,


In [9]:
%%sql

DESCRIBE DETAIL spark_catalog.deltalake.lc_ex2

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
delta,837b3cf0-2b49-4c3f-baa7-327f8e36e0c9,spark_catalog.deltalake.lc_ex2,None,s3a://warehouse/default/deltalake/lc_ex2,2026-05-20 20:32:25.956000,2026-05-20 20:55:28,[],"['invoice_date', 'customer_id']",1,557308,{},1,7,"['appendOnly', 'clustering', 'domainMetadata', 'invariants']"


In [25]:
%%sql

SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex2; 

Running query in 'SparkSession'

1 rows affected.

Field 1
99457


In [42]:
%%sql 

ALTER TABLE spark_catalog.deltalake.lc_ex2 
CLUSTER BY (invoice_date, customer_id)

Running query in 'SparkSession'

++
||
++
++

In [23]:
spark.sql('ALTER TABLE spark_catalog.deltalake.lc_ex2 CLUSTER BY (invoice_date, customer_id)')

DataFrame[]

In [26]:
%%sql 

OPTIMIZE spark_catalog.deltalake.lc_ex2 FULL

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/lc_ex2,"Row(numFilesAdded=0, numFilesRemoved=0, filesAdded=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), filesRemoved=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), partitionsOptimized=0, zOrderStats=None, clusteringStats=Row(inputZCubeFiles=Row(numFiles=1, size=557308), inputOtherFiles=Row(numFiles=0, size=0), inputNumZCubes=1, mergedFiles=Row(numFiles=0, size=0), numOutputZCubes=0), numBins=0, numBatches=1, totalConsideredFiles=1, totalFilesSkipped=1, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1779331265078, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [30]:
%%sql

SHOW CREATE TABLE spark_catalog.deltalake.lc_ex2;

Running query in 'SparkSession'

RuntimeError: [DELTA_OPERATION_NOT_ALLOWED] Operation not allowed: `SHOW CREATE TABLE` is not supported for Delta tables;
ShowCreateTable false, [createtab_stmt#3223]
+- ResolvedTable org.apache.spark.sql.delta.catalog.DeltaCatalog@74f8fb90, deltalake.lc_ex2, DeltaTableV2(org.apache.spark.sql.SparkSession@526127ad,s3a://warehouse/default/deltalake/lc_ex2,Some(CatalogTable(
Catalog: spark_catalog
Database: deltalake
Table: lc_ex2
Owner: brijeshdhaker
Created Time: Wed May 20 20:32:29 IST 2026
Last Access: UNKNOWN
Created By: Spark 3.5.3
Type: MANAGED
Provider: delta
Statistics: 557308 bytes
Location: s3a://warehouse/default/deltalake/lc_ex2
Serde Library: org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe
InputFormat: org.apache.hadoop.mapred.SequenceFileInputFormat
OutputFormat: org.apache.hadoop.hive.ql.io.HiveSequenceFileOutputFormat
Partition Provider: Catalog)),Some(deltalake.lc_ex2),None,Map()), [customer_id#3224, category#3225, price#3226, quantity#3227, invoice_date#3228]



In [ ]:
from delta.tables import *

DeltaTable.create(spark) \
  .tableName("spark_catalog.deltalake.lc_ex2") \
  .addColumn("col1", "STRING") \
  .addColumn("col2", "INT") \
  .clusterBy("col1", "col2")\
  .execute()

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '.'.(line 1, pos 23)

== SQL ==
spark_catalog.deltalake.lc_ex2
-----------------------^^^


In [43]:

#df_union.write.format("delta").mode("overwrite").clusterBy("invoice_date", "customer_id").saveAsTable("spark_catalog.deltalake.lc_ex2")
#df_union.write.format("delta").mode("overwrite").clusterBy("invoice_date", "customer_id").saveAsTable("spark_catalog.deltalake.lc_ex2")

df_union.write \
    .format("delta") \
    .mode("overwrite") \
    .clusterBy("invoice_date", "customer_id") \
    .saveAsTable("spark_catalog.deltalake.lc_ex2")

# DataFrameWriterV2
#df_union.writeTo("spark_catalog.deltalake.lc_ex2").using("delta").clusterBy("invoice_date", "customer_id").create()

AttributeError: 'DataFrameWriter' object has no attribute 'clusterBy'

In [31]:
%%sql

SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex2; 

Running query in 'SparkSession'

1 rows affected.

Field 1
99457


In [32]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex1
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 1.48 ms, sys: 371 μs, total: 1.85 ms
Wall time: 44.9 ms


DataFrame[category: string, total_sales: double]

In [33]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex2
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 1.51 ms, sys: 129 μs, total: 1.64 ms
Wall time: 35.3 ms


DataFrame[category: string, total_sales: double]